In [1]:
import pandas as pd
from enumerate import *

In [2]:
df = pd.read_csv("data/schemes.csv")
len(df)

10666

In [3]:
df.head()

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth
0,Scheme1,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,A,A,B,B,A,A
1,Scheme2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,A,A,B,B,A,B
2,Scheme3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,A,A,B,B,B,A
3,Scheme4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,A,A,B,B,B,B
4,Scheme5,1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5,A,A,B,A,A,A


In [4]:
score_columns: list[str] = [
    "husband_wife", "brother_sister",
    "brother_brother", "sister_sister",
    "son_father", "daughter_father",
    "son_mother", "daughter_mother"
]

df["score"] = df.apply(lambda row: row[score_columns].sum(), axis = 1)
df.head()

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
0,Scheme1,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,A,A,B,B,A,A,5.0
1,Scheme2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,A,A,B,B,A,B,4.0
2,Scheme3,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,A,A,B,B,B,A,4.0
3,Scheme4,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,A,A,B,B,B,B,5.0
4,Scheme5,1.0,0.5,1.0,0.5,1.0,0.5,1.0,0.5,A,A,B,A,A,A,6.0


In [5]:
df.score.max()

np.float64(6.25)

In [6]:
df.groupby("score")[["scheme"]].count().rename(columns = dict(scheme = "num_schemes"))

,num_schemes
score,
1.50,8
1.75,44
2.00,342
2.25,544
2.50,354
2.75,432
3.00,998
3.25,1560
3.50,1394


In [7]:
len(df[score_columns].drop_duplicates())

808

In [8]:
df[df.score == 6.25]

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25
7122,Scheme7123,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,B-D,C-D,B-D,B-D,B-D,6.25
7198,Scheme7199,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,C-A,C-D,C-A,C-A,C-A,6.25
9786,Scheme9787,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,D-B,C-D,D-B,D-B,D-B,6.25


In [9]:
distinct_scores: pd.DataFrame = (
    df.groupby(score_columns, as_index = False)
    .scheme.count()
    .sort_values("scheme", ascending = False) # type: ignore
)

distinct_scores.head()

,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,scheme
70,0.0,0.25,0.50,0.50,0.25,0.25,0.25,0.25,264
122,0.0,0.25,0.75,0.75,0.50,0.25,0.25,0.50,160
117,0.0,0.25,0.75,0.75,0.25,0.50,0.50,0.25,160
269,0.0,0.50,1.00,1.00,0.50,0.50,0.50,0.50,144
212,0.0,0.50,0.75,0.75,0.50,0.50,0.25,0.25,96


In [10]:
distinct_scores.scheme.describe()

count    808.000000
mean      13.200495
std       19.352451
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      264.000000
Name: scheme, dtype: float64

In [11]:
print(list(enumerate(distinct_scores.scheme.head(30))))

[(0, 264), (1, 160), (2, 160), (3, 144), (4, 96), (5, 96), (6, 76), (7, 76), (8, 76), (9, 76), (10, 72), (11, 64), (12, 64), (13, 64), (14, 64), (15, 64), (16, 64), (17, 64), (18, 64), (19, 64), (20, 64), (21, 64), (22, 64), (23, 64), (24, 64), (25, 64), (26, 64), (27, 60), (28, 60), (29, 56)]


In [12]:
schemes: list[type[Scheme]] = \
    import_schemes_from_dataframe(df.loc[:, Family._fields], df.scheme) # type: ignore

[scheme.__name__ for scheme in schemes[:5]]

['Scheme1', 'Scheme2', 'Scheme3', 'Scheme4', 'Scheme5']

In [13]:
target_family = Family(
    father_birth = "A-B",
    father_married = "A-D",
    mother_birth = "C-D",
    mother_married = "A-D",
    son_birth = "A-D",
    daughter_birth = "A-D"
)

if any(
    (target_index := i) == i and export_scheme_to_family(scheme) == target_family
    for i, scheme in enumerate(schemes)
):
    print(target_index)

5154


In [14]:
df.iloc[target_index]

scheme             Scheme5155
husband_wife              1.0
brother_sister            0.5
brother_brother          0.75
sister_sister            0.75
son_father               0.75
daughter_father          0.75
son_mother               0.75
daughter_mother          0.75
father_birth              A-B
father_married            A-D
mother_birth              C-D
mother_married            A-D
son_birth                 A-D
daughter_birth            A-D
score                     6.0
Name: 5154, dtype: object

In [15]:
df[df.score == 6]

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
4,Scheme5,1.0,0.5,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.0
5,Scheme6,1.0,0.5,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.0
4162,Scheme4163,1.0,0.5,1.00,0.50,1.00,0.50,1.00,0.50,A-B,A-B,C-D,A-B,A-B,A-B,6.0
5154,Scheme5155,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-D,C-D,A-D,A-D,A-D,6.0
6502,Scheme6503,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,B-C,C-D,B-C,B-C,B-C,6.0
7818,Scheme7819,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,C-B,C-D,C-B,C-B,C-B,6.0
8330,Scheme8331,1.0,0.5,0.50,1.00,0.50,1.00,0.50,1.00,A-B,C-D,C-D,C-D,C-D,C-D,6.0
9166,Scheme9167,1.0,0.5,0.75,0.75,0.75,0.75,0.75,0.75,A-B,D-A,C-D,D-A,D-A,D-A,6.0


In [16]:
from schemes import *

schemes: set[type[Scheme]] = {
    value for value in globals().values()
    if isinstance(value, type) and issubclass(value, Scheme) and value is not Scheme
    and not value.__name__.startswith("Scheme_")
}

schemes

{schemes.AlphabeticalPriority,
 schemes.AnglophoneTraditional,
 schemes.ChineseTraditional,
 schemes.DoubleSurname,
 schemes.DualLineage,
 schemes.JointFamily,
 schemes.Matrilineal,
 schemes.MatrilinealMarriage,
 schemes.SpanishTraditional,
 schemes.StarWars,
 schemes.Symmetric}

In [17]:
COLUMNS: list[str] = list(Family._fields)
VALUES: list[list[str]] = [list(export_scheme_to_family(scheme)) for scheme in schemes]

manual_schemes: pd.DataFrame = df[df.set_index(COLUMNS).index.isin(VALUES)]
manual_schemes

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score
0,Scheme1,0.0,1.00,1.00,1.00,1.00,1.00,0.00,0.00,A,A,B,B,A,A,5.00
1,Scheme2,0.0,0.00,1.00,1.00,1.00,0.00,0.00,1.00,A,A,B,B,A,B,4.00
3,Scheme4,0.0,1.00,1.00,1.00,0.00,0.00,1.00,1.00,A,A,B,B,B,B,5.00
4,Scheme5,1.0,0.50,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.00
5,Scheme6,1.0,0.50,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.00
10,Scheme11,0.5,0.75,1.00,0.75,1.00,0.75,0.50,0.50,A,A,B,B-A,A,A,5.75
3671,Scheme3672,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-C,A-C,5.00
3684,Scheme3685,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-D,A-D,5.00
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25


In [18]:
manual_schemes["manual"] = [[] for i in range(len(manual_schemes))]

for i in manual_schemes.index:
    row_scheme: type[Scheme] = import_scheme_from_series(manual_schemes.loc[i, COLUMNS]) # type: ignore
    for scheme in schemes:
        if export_scheme_to_family(scheme) == export_scheme_to_family(row_scheme):
            manual_schemes.loc[i, "manual"].append(scheme.__name__)

manual_schemes

,scheme,husband_wife,brother_sister,brother_brother,sister_sister,son_father,daughter_father,son_mother,daughter_mother,father_birth,father_married,mother_birth,mother_married,son_birth,daughter_birth,score,manual
0,Scheme1,0.0,1.00,1.00,1.00,1.00,1.00,0.00,0.00,A,A,B,B,A,A,5.00,[ChineseTraditional]
1,Scheme2,0.0,0.00,1.00,1.00,1.00,0.00,0.00,1.00,A,A,B,B,A,B,4.00,[Symmetric]
3,Scheme4,0.0,1.00,1.00,1.00,0.00,0.00,1.00,1.00,A,A,B,B,B,B,5.00,[Matrilineal]
4,Scheme5,1.0,0.50,1.00,0.50,1.00,0.50,1.00,0.50,A,A,B,A,A,A,6.00,[AnglophoneTraditional]
5,Scheme6,1.0,0.50,0.50,1.00,0.50,1.00,0.50,1.00,A,B,B,B,B,B,6.00,[MatrilinealMarriage]
10,Scheme11,0.5,0.75,1.00,0.75,1.00,0.75,0.50,0.50,A,A,B,B-A,A,A,5.75,[StarWars]
3671,Scheme3672,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-C,A-C,5.00,[SpanishTraditional]
3684,Scheme3685,0.0,1.00,1.00,1.00,0.50,0.50,0.50,0.50,A-B,A-B,C-D,C-D,A-D,A-D,5.00,[DualLineage]
4534,Scheme4535,1.0,0.75,0.75,0.75,0.75,0.75,0.75,0.75,A-B,A-C,C-D,A-C,A-C,A-C,6.25,"[AlphabeticalPriority, JointFamily]"
